In [1]:
import pandas as pd
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

In this notebook, we introduce the increased mortality among people actively suffering from TB disease. In our model, we don't have any screening or treatment, so we stick to older estimates of mortality before antibiotics. 

Some estimates suggest that 50% of untreated patients died within 2-5 years, so let's say the TB specific mortality rate from the infectious compartment is 0.14, which is added to the universal death rate here.

In [15]:
def build_seirs_model(
        config: dict,
) -> CompartmentalModel:
    """ 
    Args: 
        config: dict with configs other than params
    Returns:
        model: the summer model obejct
    """

    # For now we just keep the same compartments, ie. we don't define a new susceptible compartment for recovered people
    compartments = (
        "susceptible",
        "exposed",
        "infectious",
        "recovered",
    )
    analysis_times = (config["start_time"], config["end_time"])

    model = CompartmentalModel(
        times = analysis_times,
        compartments = compartments,
        infectious_compartments = ("infectious",),
    )

    model.set_initial_population(
        distribution={
            "susceptible": config["population"] - config["seed"],
            "infectious": config["seed"],
        },
    )
    
    model.add_crude_birth_flow(
        "births",
        Parameter("birth_rate"),
        "susceptible",
    )

    model.add_universal_death_flows(
        "background_mortality",
        death_rate=Parameter("death_rate",)
    )

    model.add_infection_frequency_flow(
        name="infection",
        contact_rate=Parameter("contact_rate"),
        source="susceptible",
        dest="exposed",
    )
    model.add_transition_flow(
        name="progression",
        fractional_rate=Parameter("progression"),
        source="exposed",
        dest="infectious",
    )
    model.add_transition_flow(
        name="recovery",
        fractional_rate=Parameter("recovery"),
        source="infectious",
        dest="recovered",
    )

    model.add_transition_flow(
        name="natural_clearance",
        fractional_rate=Parameter("clearance_rate"),
        source="exposed",
        dest="recovered",
    )

    model.add_transition_flow(
        name="waning_immunity",
        fractional_rate=Parameter("waning_immunity"),
        source="recovered",
        dest="susceptible",                
    )

    # TB specific death rate:
    model.add_death_flow(
        name="tb_mortality",
        death_rate=Parameter("tb_death_rate"),
        source="infectious"
    )

    return model

In [16]:
# We use the same configurations:
model_config = {
    "population": 48928.0, # TB study pop in 2004 in Guinea-Bissau, we assume everyone is susceptible if not infected
    "seed": 144.0, # Number of cases diagnosed in 2004 - the seed
    "start_time": 2004,
    "end_time": 2020,
}

# Create instance of SEIRS model with these configs
seirs_model = build_seirs_model(model_config)

In [17]:
# We add the death rate
parameters = {
    "recovery": 0.15,
    "contact_rate": 10.0,
    "death_rate": 0.022, # Average life expectancy was 45.5 in 2004
    "birth_rate": 0.022, # We start by matching the death rate to keep the population stable over time
    "tb_death_rate": 0.14,

    "clearance_rate": 0.8, # Say 80% of infected peolpe actually recover naturally
    "progression": 0.2, # Should then be 20% to give the average dwell time in the latent compartment of 1 year
    "waning_immunity": 0.0414
}

In [18]:
# Run and plot:
seirs_model.run(parameters=parameters)
compartment_values = seirs_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)

Number of infectious individuals is still too high, but slowly getting closer to something realistic. the population is also getting smaller, because the overall death rate is now higher than the overall birth date. In Guinea-Bissau, the population has been increasing over the period, so we should include that.

We're also still assuming that the non-infectious population in 2004 were all susceptible, and we're assuming no vaccine- or treatment coverage. 